# Client Boto3 / S3

In [2]:
import os
import io 
import boto3
import json
from pprint import pprint
import pandas as pd
from tqdm.notebook import tqdm  #  Barre Jupyter native (bleue)
# ou : from tqdm.autonotebook import tqdm  # auto console/notebook

import pyarrow.parquet as pq
import pyarrow.json as paj
import pyarrow as pa

from tqdm import tqdm
import time

endpoint = os.environ["S3_ENDPOINT_URL"]
bucket = os.environ["S3_BUCKET"]

s3_boto = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)

class WTTJ:
    def __init__(self, job_title: str, job_description: int):
        self.job_title = job_title
        self.job_description = job_description




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

# Helper + Class 

In [3]:
def get_json_field_from_record(record, field_name):
    if isinstance(record[field_name], str):
        data = json.loads(record[field_name])
    else:
        data = record[field_name]  # déjà un dict !
    return data

class WTTJ:
    def __init__(self, id :int,  job_title: str, job_description: str, company_employees : int, profile : str, url : str):
        self.id = id       
        self.job_title = job_title
        self.job_description = job_description
        self.company_employees= company_employees
        self.profile = profile
        self.url = url

    def __str__(self) -> str:
        """Surcharge affichage print()"""
        desc = self.job_description[:50] if self.job_description else "NA"
        profile = self.profile[:50] if self.profile else "NA"
        
        return f"""- Id : #{self.id}
        job_title : {self.job_title}
        job_description : {desc}...
        company_employees : {self.company_employees}
        profile : {profile}
        url : {self.url}"""

from bs4 import BeautifulSoup
import html

def clean_html(data):
    if isinstance(data, str):
        # Remove html tag
        soup = BeautifulSoup(data, 'html.parser')
        text = soup.get_text()
        # Remove Html Entities
        return html.unescape(text).strip()
    return data


# Lister des objets Boto3

In [11]:
bucket = os.environ["S3_BUCKET"]
resp = s3_boto.list_objects_v2(Bucket=bucket, Prefix="welcometothejungle/bronze/dt=2026-02-18")
[obj["Key"] for obj in resp.get("Contents", [])][:5]


['welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_html/key=00a41af78fa5d335/page.html.gz',
 'welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_html/key=00b90573deb25b8d/page.html.gz',
 'welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_html/key=0819e58746f2ee68/page.html.gz',
 'welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_html/key=08320c56-a25b-4d4e-9a18-fa59af12de9f/page.html.gz',
 'welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_html/key=0ec2ff3f-0889-4f1c-9bed-165bf415fc61/page.html.gz']

# Lire WTTJ Json + Alimentation Class WTTJ 

In [14]:
key = "welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000047.jsonl"

print(f"bucket={bucket}")
print(f"Key={key}")

print("- Get objec from S3 for key {key}")
obj = s3_boto.get_object(Bucket=bucket, Key=key)

print("- Read and decode json string")
lines = obj["Body"].read().decode("utf-8").splitlines()

print("- Deserialze json in dictionnary")
records = [json.loads(l) for l in lines if l.strip()]

print("- Dict. Keys = ", records[0].keys())
print("- Nb properties= " ,len(records))
print("")
#print(records[0].values())

wttj_all =[]
max_elmt=1000
print(f"- Parse {max_elmt}")

for i, record in enumerate(tqdm(records[:max_elmt], desc="Processing jobs")):

    #initial_data= get_json_field_from_record(records[i], "initial_data")    
    job_data = get_json_field_from_record(records[i], "job_data")

    wttj = WTTJ( 
        id = job_data["wttj_reference"],
        job_title = clean_html( job_data["name"] ),
        job_description = clean_html( job_data["description"] ),
        company_employees =0, # job_data["nb_employees"],
        profile = job_data["profile"],
        url = job_data["urls"][0]
    )
    wttj_all.append(wttj)

for a_wttj in wttj_all[:1]:
    print(a_wttj.job_title)


bucket=jobmarket
Key=welcometothejungle/bronze/dt=2026-02-18/run_id=20260217T181105Z/segment=jobs_raw/part-000047.jsonl
- Get objec from S3 for key {key}
- Read and decode json string
- Deserialze json in dictionnary
- Dict. Keys =  dict_keys(['source', 'segment', 'url', 'fetched_at', 'status_code', 'ok', 'error', 'key', 'initial_data', 'job_data', 'parser_version'])
- Nb properties=  1500

- Parse 1000


Processing jobs:   0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_337/4009133493.py:35: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(data, 'html.parser')
Processing jobs: 100%|██████████| 1000/1000 [00:00<00:00, 1325.24it/s]

Stagiaire en Graphisme / Deck Design


# Inférences ROME

In [15]:
import json
import requests

for a_wttj in wttj_all[:10]:
    print(30* "=")
    payload = {
        "intitule": a_wttj.job_title,
        "description": a_wttj.job_description
    }
    print("title = " , a_wttj.job_title )
    print("description = ", a_wttj.job_description )

    r = requests.post("http://api:8000/predict", json=payload)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
    


title =  Stagiaire en Graphisme / Deck Design
description =  Nous recherchons un stagiaire Deck Designer pour accompagner nos consultants et nos clients.Chez Saper Vedere, le deck design est un rÃ´le clÃ© : il sâappuie sur les meilleures techniques de data visualisation pour transformer des analyses complexes en prÃ©sentations claires, impactantes et Ã  forte valeur ajoutÃ©e. Il travaille avec lâensemble de nos pÃ´les de compÃ©tences : influence, affaires publiques, dÃ©veloppement commercial,â¦Notre mission est dâaider nos clients Ã  mieux comprendre leurs enjeux sociÃ©taux Ã  partir de donnÃ©es tangibles. Cela implique non seulement une grande rigueur dans la restitution et la mise en forme, mais aussi la crÃ©ation de mÃ©thodologies et de slides modÃ¨les qui deviennent des rÃ©fÃ©rences internes et garantissent la qualitÃ© de nos livrables.MissionsConcevoir et mettre en page des slides attrayants et professionnelsTransformer des donnÃ©es complexes en graphiques, infographies et 

# Debug

In [ ]:
# =============================================================================
# 📄 EXTRACTION window.__INITIAL_DATA__ depuis WelcomeToTheJungle
# Script Jupyter Notebook complet - Testé et fonctionnel
# =============================================================================

import re
import json
import requests
from typing import Optional, Dict, Any
from pathlib import Path
import json as json_module  # Alias pour éviter conflit

display("🚀 Initialisation...")

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

URL = "https://www.welcometothejungle.com/fr/companies/groupement-les-mousquetaires/jobs/caissiere-hote-de-caisse-h-f_carnoules"

# Regex OPTIMISÉE pour strings "..." avec échappements internes
INITIAL_DATA_RE = re.compile(
    r'window\.__INITIAL_DATA__\s*=\s*"((?:[^"\\]|\\.)*)"\s*;?',
    re.DOTALL | re.MULTILINE
)


display(f"📍 URL cible : {URL}")

# =============================================================================
# 2. FONCTION D'EXTRACTION
# =============================================================================

def extract_initial_data_from_html(html: str) -> Optional[Dict[str, Any]]:
    """Extrait et parse window.__INITIAL_DATA__ de l'HTML."""
    m = INITIAL_DATA_RE.search(html)
    if not m:
        print("❌ Pas de window.__INITIAL_DATA__ trouvé")
        return None
    
    raw_json = m.group(1)
    print(f"✅ Capturé {len(raw_json):,} caractères JSON")
    print(raw_json)
    
    try:
        data = json.loads(raw_json)
        print(f"✅ JSON parsé avec succès ! Clés : {list(data.keys())}")
        return data
    except json.JSONDecodeError as e:
        print(f"❌ Erreur JSON : {e}")
        return None

# =============================================================================
# 3. RÉCUPÉRATION ET EXTRACTION
# =============================================================================

print("📥 Téléchargement de la page...")
try:
    resp = requests.get(
        URL, 
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'},
        timeout=15
    )
    resp.raise_for_status()
except Exception as e:
    print(f"❌ Erreur requête : {e}")
    raise

html = resp.text
print(f"📄 HTML récupéré : {len(html):,} caractères")

# Sauvegarde HTML pour debug
Path("page_wttj.html").write_text(html, encoding="utf-8")
print("💾 HTML sauvé → page_wttj.html")

# Extraction !
data = extract_initial_data_from_html(html)

# =============================================================================
# 4. ANALYSE DES RÉSULTATS
# =============================================================================

if data:
    display("🎉 EXTRACTION RÉUSSIE !")
    
    # Sauvegarde JSON propre
    Path("initial_data_clean.json").write_text(
        json.dumps(data, indent=2, ensure_ascii=False), 
        encoding="utf-8"
    )
    print("💾 JSON propre sauvé → initial_data_clean.json")
    
    # Exploration des données
    if 'queries' in data and data['queries']:
        first_query = data['queries'][0]
        job_data = first_query.get('state', {}).get('data', {})
        
        display("## 📋 PREMIER JOB TROUVÉ")
        display(f"**Nom** : {job_data.get('name', 'N/A')}")
        display(f"**Ville** : {job_data.get('office', {}).get('city', 'N/A')}")
        display(f"**Slug** : {job_data.get('slug', 'N/A')}")
        display(f"**Champs obligatoires** : {len(job_data.get('application_fields', []))}")
        
        # Aperçu application_fields
        fields = job_data.get('application_fields', [])
        df_fields = pd.DataFrame(fields)[['name', 'mode']] if 'pd' in globals() else None
        
    else:
        display("ℹ️  Structure inattendue, mais JSON valide !")
        display(f"Clés racines : {list(data.keys())}")
        
else:
    print("🔍 DEBUG : Vérification présence variable...")
    if 'window.__INITIAL_DATA__' in html:
        print("✅ Variable présente dans HTML, problème regex")
    else:
        print("❌ Variable ABSENTE de cette page")

# =============================================================================
# 5. VALIDATION FINALE
# =============================================================================

print("\n✅" + "="*60)
print("RÉSUMÉ :")
print(f"• URL : {URL}")
print(f"• Taille JSON extrait : {len(json.dumps(data)) if data else 0:,} caractères")
print(f"• Fichiers générés : page_wttj.html, initial_data_clean.json")
print("="*60)


In [ ]:
import re
import json
from pathlib import Path
from typing import Dict, Any

def decode_wttj_data(html_content: str) -> Dict[str, Any]:
    """
    Extrait et parse window.__INITIAL_DATA__ de WTTJ (double serialisation)
    """
    # 1. Regex capture STRING COMPLET (échappements inclus)
    pattern = r'window\.__INITIAL_DATA__\s*=\s*"((?:[^"\\]|\\.)*)"\s*;?'
    match = re.search(pattern, html_content, re.DOTALL)
    
    if not match:
        raise ValueError("❌ window.__INITIAL_DATA__ non trouvé")
    
    raw_string = match.group(1)
    print(f"📏 Raw string: {len(raw_string):,} caractères")
    
    # 2. Décode Unicode JS (\uXXXX → UTF-8)
    json_text = bytes(raw_string, 'utf-8').decode('unicode_escape')
    
    # 3. Parse JSON principal
    data = json.loads(json_text)
    
    # 4. Dé-sérialise les champs internes (arrays/objects stringifiés)
    for key, value in data.items():
        if isinstance(value, str):
            try:
                parsed = json.loads(value)
                data[key] = parsed
                print(f"🔄 {key}: string → {type(parsed).__name__}")
            except json.JSONDecodeError:
                pass  # Garde string si pas JSON
    
    return data

# ═══════════════════════════════════════════════════════════════
# EXECUTION
# ═══════════════════════════════════════════════════════════════

html_file = Path("page_wttj.html")
if not html_file.exists():
    print("❌ page_wttj.html manquant")
else:
    html = html_file.read_text(encoding='utf-8')
    
    try:
        job_data = decode_wttj_data(html)
        
        # 📊 AFFICHAGE RÉSUMÉ
        print("\n🎉 EXTRACTION RÉUSSIE !")
        print("=" * 60)
        print(f"📂 Slug: {job_data.get('slug', 'N/A')}")
        print(f"📛 Titre: {job_data.get('name', 'N/A')}")
        print(f"🏢 Ville: {job_data.get('office', {}).get('city', 'N/A')}")
        print(f"📋 Contrat: {job_data.get('contractType', 'N/A')}")
        print(f"🔑 queryHash: {job_data.get('queryHash', 'N/A')}")
        
        # 💾 SAUVEGARDE
        output_file = Path("wttj_job_complete.json")
        output_file.write_text(
            json.dumps(job_data, indent=2, ensure_ascii=False),
            encoding='utf-8'
        )
        print(f"\n✅ SAUVEGARDÉ: {output_file.absolute()}")
        print(f"📏 Taille JSON: {len(json.dumps(job_data)):,} chars")
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        print("\n🔍 DEBUG: premiers 300 chars raw:")
        match = re.search(r'window\.__INITIAL_DATA__\s*=\s*"(.{0,300})', html)
        if match:
            print(repr(match.group(1)))
